# Inverse dynamica - Pantograaf

Deze notebook gebruikt dezelfde geometrische parameters en dezelfde opgelegde inputbeweging als `Notebook_Matteo_Henry.ipynb`, maar start inhoudelijk bij de inverse dynamica.

De volgorde blijft technisch noodzakelijk:

1. leg de beweging `theta1(t)` op;
2. los de kinematica op zodat posities, snelheden en versnellingen gekend zijn;
3. stel per bewegend gelid de Newton-Euler vergelijkingen op;
4. los per tijdstap het lineaire stelsel `A_dyn w = B_dyn` op.

De onbekenden in `w` zijn alle scharnierreacties en het aandrijfmoment rond punt `A`.

## 1. Gekende beweging en gedeelde parameters

Inverse dynamica vertrekt vanuit een gekende beweging. Hier leggen we dezelfde relatieve inputhoek op als in de hoofdnotebook:

$$
\theta_1(t)=A_{drive}\cos(\omega_{drive}t),
$$

met `A_drive = 40 deg` en `omega_drive = 0.32 rad/s`. De afhankelijke hoeken volgen uit dezelfde drie sluitingslussen als in de andere notebook.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import fsolve

# Zelfde geometrische parameters als in Notebook_Matteo_Henry.ipynb.
r1, r2, r3, r6 = 0.18, 0.24, 0.16, 0.17
l41, l42 = 0.17, 0.25
l51, l52 = 0.13, 0.15
l71, l72 = 0.12, 0.40

d1, d2, d3 = 0.20, 0.16, 0.11

A = np.array([0.0, 0.0])
B = np.array([d1, 0.0])
E = np.array([d1 + d2, -d3])

theta1_zero_abs = np.deg2rad(270.0)
x_init_abs = np.deg2rad([-80, -29, 90, -41, -15, -64])


def e(theta, L=1.0):
    return np.array([L*np.cos(theta), L*np.sin(theta)])


def e_arr(theta, L=1.0):
    return np.column_stack((L*np.cos(theta), L*np.sin(theta)))


def theta1_to_abs(theta1_rel):
    return theta1_zero_abs + theta1_rel


def loop_closure_eqs(x_abs, theta1_abs_val):
    theta2_abs, theta4_abs, theta3_abs, theta5_abs, theta6_abs, theta7_abs = x_abs

    loop1 = (B - A) + e(theta2_abs, r2) - e(theta4_abs, l41) - e(theta1_abs_val, r1)
    loop2 = (E - B) + e(theta5_abs, l51) - e(theta3_abs, r3) - e(theta4_abs, l42) - e(theta2_abs, r2)
    loop3 = -e(theta3_abs, r3) + e(theta6_abs, r6) - e(theta7_abs, l71) - e(theta5_abs, l52)

    return np.array([loop1[0], loop1[1], loop2[0], loop2[1], loop3[0], loop3[1]])


def solve_pose(theta1_abs_val, x_guess_abs):
    sol, info, ier, msg = fsolve(
        lambda x: loop_closure_eqs(x, theta1_abs_val),
        x_guess_abs,
        full_output=True,
        xtol=1e-11,
        maxfev=2000,
    )
    return sol, ier, msg


def reconstruct_points(theta1_abs_val, theta2_abs_val, theta4_abs_val, theta3_abs_val, theta5_abs_val, theta6_abs_val, theta7_abs_val):
    C = A + e(theta1_abs_val, r1)
    D = B + e(theta2_abs_val, r2)
    G = D + e(theta4_abs_val, l42)
    F = G + e(theta3_abs_val, r3)
    H = E + e(theta5_abs_val, l51 + l52)
    I = G + e(theta6_abs_val, r6)
    End = H + e(theta7_abs_val, l71 + l72)
    return C, D, G, F, H, I, End

# Nulconfiguratie gebruiken als stabiele startschatting voor de tijdreeks.
x_zero_abs, ier_zero, msg_zero = solve_pose(theta1_to_abs(0.0), x_init_abs)
if ier_zero != 1:
    raise RuntimeError('Nulconfiguratie niet geconvergeerd: ' + msg_zero)

print('Gedeelde parameters geladen.')
print('Nulconfiguratie voor afhankelijke hoeken [deg]:')
print(np.round(np.rad2deg(x_zero_abs), 3))

In [ ]:
def loop_jacobian(x):
    theta2_val, theta4_val, theta3_val, theta5_val, theta6_val, theta7_val = x
    return np.array([
        [-r2*np.sin(theta2_val),  l41*np.sin(theta4_val), 0.0, 0.0, 0.0, 0.0],
        [ r2*np.cos(theta2_val), -l41*np.cos(theta4_val), 0.0, 0.0, 0.0, 0.0],
        [ r2*np.sin(theta2_val),  l42*np.sin(theta4_val),  r3*np.sin(theta3_val), -l51*np.sin(theta5_val), 0.0, 0.0],
        [-r2*np.cos(theta2_val), -l42*np.cos(theta4_val), -r3*np.cos(theta3_val),  l51*np.cos(theta5_val), 0.0, 0.0],
        [0.0, 0.0,  r3*np.sin(theta3_val),  l52*np.sin(theta5_val), -r6*np.sin(theta6_val),  l71*np.sin(theta7_val)],
        [0.0, 0.0, -r3*np.cos(theta3_val), -l52*np.cos(theta5_val),  r6*np.cos(theta6_val), -l71*np.cos(theta7_val)],
    ], dtype=float)


def rhs_position_derivative(theta1_val):
    return np.array([
        -r1*np.sin(theta1_val),
         r1*np.cos(theta1_val),
         0.0,
         0.0,
         0.0,
         0.0,
    ], dtype=float)


def rhs_velocity(theta1_val, dtheta1_val):
    return rhs_position_derivative(theta1_val) * dtheta1_val


def rhs_acceleration(theta1_val, dtheta1_val, ddtheta1_val):
    return np.array([
        -r1*np.sin(theta1_val)*ddtheta1_val - r1*np.cos(theta1_val)*dtheta1_val**2,
         r1*np.cos(theta1_val)*ddtheta1_val - r1*np.sin(theta1_val)*dtheta1_val**2,
         0.0,
         0.0,
         0.0,
         0.0,
    ], dtype=float)


def jacobian_dot_times_qdot(x, qdot):
    theta2_val, theta4_val, theta3_val, theta5_val, theta6_val, theta7_val = x
    dtheta2_val, dtheta4_val, dtheta3_val, dtheta5_val, dtheta6_val, dtheta7_val = qdot
    return np.array([
        -r2*np.cos(theta2_val)*dtheta2_val**2 + l41*np.cos(theta4_val)*dtheta4_val**2,
        -r2*np.sin(theta2_val)*dtheta2_val**2 + l41*np.sin(theta4_val)*dtheta4_val**2,
         r2*np.cos(theta2_val)*dtheta2_val**2 + l42*np.cos(theta4_val)*dtheta4_val**2 + r3*np.cos(theta3_val)*dtheta3_val**2 - l51*np.cos(theta5_val)*dtheta5_val**2,
         r2*np.sin(theta2_val)*dtheta2_val**2 + l42*np.sin(theta4_val)*dtheta4_val**2 + r3*np.sin(theta3_val)*dtheta3_val**2 - l51*np.sin(theta5_val)*dtheta5_val**2,
         r3*np.cos(theta3_val)*dtheta3_val**2 + l52*np.cos(theta5_val)*dtheta5_val**2 - r6*np.cos(theta6_val)*dtheta6_val**2 + l71*np.cos(theta7_val)*dtheta7_val**2,
         r3*np.sin(theta3_val)*dtheta3_val**2 + l52*np.sin(theta5_val)*dtheta5_val**2 - r6*np.sin(theta6_val)*dtheta6_val**2 + l71*np.sin(theta7_val)*dtheta7_val**2,
    ], dtype=float)


def link_velocity(theta_arr, L, dtheta_arr):
    return np.column_stack((-L*np.sin(theta_arr)*dtheta_arr, L*np.cos(theta_arr)*dtheta_arr))


def link_acceleration(theta_arr, L, dtheta_arr, ddtheta_arr):
    return np.column_stack((
        -L*np.sin(theta_arr)*ddtheta_arr - L*np.cos(theta_arr)*dtheta_arr**2,
         L*np.cos(theta_arr)*ddtheta_arr - L*np.sin(theta_arr)*dtheta_arr**2,
    ))


def point_from_base(base_pos, base_vel, base_acc, theta_arr, L, dtheta_arr, ddtheta_arr):
    pos = base_pos + e_arr(theta_arr, L)
    vel = base_vel + link_velocity(theta_arr, L, dtheta_arr)
    acc = base_acc + link_acceleration(theta_arr, L, dtheta_arr, ddtheta_arr)
    return pos, vel, acc

In [ ]:
# Opgelegde beweging, identiek aan de hoofdnotebook.
t_begin = 0.0
t_end = 20.0
Ts = 0.05
t = np.arange(t_begin, t_end + Ts, Ts)

theta_mean = np.deg2rad(0.0)
A_drive = np.deg2rad(40.0)
omega_drive = 0.32

theta1 = theta_mean + A_drive*np.cos(omega_drive*t)
dtheta1 = -omega_drive*A_drive*np.sin(omega_drive*t)
ddtheta1 = -omega_drive**2*A_drive*np.cos(omega_drive*t)
theta1_abs = theta1_to_abs(theta1)

# Positie-analyse.
X = np.zeros((len(t), 6))
x_guess = x_zero_abs.copy()
for k in range(len(t)):
    sol, ier, msg = solve_pose(theta1_abs[k], x_guess)
    if ier != 1:
        raise RuntimeError(f'Positie-oplossing niet geconvergeerd op stap {k}: {msg}')
    X[k] = sol
    x_guess = sol.copy()

theta2_abs, theta4_abs, theta3_abs, theta5_abs, theta6_abs, theta7_abs = X.T

# Snelheids- en versnellingsanalyse uit de afgeleide sluitingsvergelijkingen.
X_q = np.zeros_like(X)
X_dot = np.zeros_like(X)
X_ddot = np.zeros_like(X)
velocity_residuals = np.zeros_like(X)
acceleration_residuals = np.zeros_like(X)

for k in range(len(t)):
    J_k = loop_jacobian(X[k])
    X_q[k] = np.linalg.solve(J_k, rhs_position_derivative(theta1_abs[k]))
    X_dot[k] = np.linalg.solve(J_k, rhs_velocity(theta1_abs[k], dtheta1[k]))
    jdot_qdot = jacobian_dot_times_qdot(X[k], X_dot[k])
    X_ddot[k] = np.linalg.solve(J_k, rhs_acceleration(theta1_abs[k], dtheta1[k], ddtheta1[k]) - jdot_qdot)

    velocity_residuals[k] = J_k @ X_dot[k] - rhs_velocity(theta1_abs[k], dtheta1[k])
    acceleration_residuals[k] = J_k @ X_ddot[k] + jdot_qdot - rhs_acceleration(theta1_abs[k], dtheta1[k], ddtheta1[k])

dtheta2, dtheta4, dtheta3, dtheta5, dtheta6, dtheta7 = X_dot.T
ddtheta2, ddtheta4, ddtheta3, ddtheta5, ddtheta6, ddtheta7 = X_ddot.T

# Punten reconstrueren.
C_arr = A + e_arr(theta1_abs, r1)
D_arr = B + e_arr(theta2_abs, r2)
G_arr = D_arr + e_arr(theta4_abs, l42)
F_arr = G_arr + e_arr(theta3_abs, r3)
H_arr = E + e_arr(theta5_abs, l51 + l52)
I_arr = G_arr + e_arr(theta6_abs, r6)
End_arr = H_arr + e_arr(theta7_abs, l71 + l72)

zero = np.zeros((len(t), 2))
C_vel = link_velocity(theta1_abs, r1, dtheta1)
C_acc = link_acceleration(theta1_abs, r1, dtheta1, ddtheta1)
D_vel = link_velocity(theta2_abs, r2, dtheta2)
D_acc = link_acceleration(theta2_abs, r2, dtheta2, ddtheta2)
G_vel = D_vel + link_velocity(theta4_abs, l42, dtheta4)
G_acc = D_acc + link_acceleration(theta4_abs, l42, dtheta4, ddtheta4)
F_vel = G_vel + link_velocity(theta3_abs, r3, dtheta3)
F_acc = G_acc + link_acceleration(theta3_abs, r3, dtheta3, ddtheta3)
H_vel = link_velocity(theta5_abs, l51 + l52, dtheta5)
H_acc = link_acceleration(theta5_abs, l51 + l52, dtheta5, ddtheta5)
I_vel = G_vel + link_velocity(theta6_abs, r6, dtheta6)
I_acc = G_acc + link_acceleration(theta6_abs, r6, dtheta6, ddtheta6)
End_vel = H_vel + link_velocity(theta7_abs, l71 + l72, dtheta7)
End_acc = H_acc + link_acceleration(theta7_abs, l71 + l72, dtheta7, ddtheta7)

print('Kinematica voor inverse dynamica berekend.')
print(f'- aantal tijdstappen = {len(t)}')
print(f'- max residu snelheid = {np.linalg.norm(velocity_residuals, axis=1).max():.3e}')
print(f'- max residu versnelling = {np.linalg.norm(acceleration_residuals, axis=1).max():.3e}')

## 2. Matrixmethode voor inverse dynamica

Voor elk bewegend gelid gebruiken we drie vergelijkingen:

$$
\sum F_x = m a_x, \qquad \sum F_y = m a_y, \qquad \sum M_{cg} = I_{cg}\alpha.
$$

Het mechanisme heeft 7 bewegende gelederen. Dat geeft 21 vergelijkingen. De 21 onbekenden zijn:

- 20 reactiekrachtcomponenten in de 10 equivalente 1-DOF gewrichten;
- 1 aandrijfmoment `M_A` rond de inputscharnier `A`.

Voor het ternaire gewricht in `G` gebruiken we twee onafhankelijke krachtparen: `G3` tussen link 4 en link 3, en `G6` tussen link 4 en link 6.

In [ ]:
# Dynamische parameters.
g = 9.81
rho_line = 8.0  # kg/m, zelfde eenvoudige lijnmassa als de hoofdnotebook

# Externe kracht op het eindpunt. Voor een treinpantograaf kan dit de neerwaartse
# contactkracht van de bovenleiding op de pantograaf voorstellen.
# Zet op [0, 0] als je alleen inertie + zwaartekracht wil tonen.
F_contact_end = np.array([0.0, -15.0])  # N

body_length = {
    '1': r1,
    '2': r2,
    '4': l41 + l42,
    '3': r3,
    '5': l51 + l52,
    '6': r6,
    '7': l71 + l72,
}
body_mass = {name: rho_line*L for name, L in body_length.items()}
body_inertia = {name: body_mass[name]*body_length[name]**2/12.0 for name in body_length}

# Massacentra, snelheden en versnellingen van de 7 bewegende gelederen.
cg_pos = {}
cg_vel = {}
cg_acc = {}
body_omega = {}
body_alpha = {}

cg_pos['1'], cg_vel['1'], cg_acc['1'] = point_from_base(np.tile(A, (len(t), 1)), zero, zero, theta1_abs, r1/2, dtheta1, ddtheta1)
cg_pos['2'], cg_vel['2'], cg_acc['2'] = point_from_base(np.tile(B, (len(t), 1)), zero, zero, theta2_abs, r2/2, dtheta2, ddtheta2)
cg_pos['4'], cg_vel['4'], cg_acc['4'] = point_from_base(C_arr, C_vel, C_acc, theta4_abs, (l41 + l42)/2, dtheta4, ddtheta4)
cg_pos['3'], cg_vel['3'], cg_acc['3'] = point_from_base(G_arr, G_vel, G_acc, theta3_abs, r3/2, dtheta3, ddtheta3)
cg_pos['5'], cg_vel['5'], cg_acc['5'] = point_from_base(np.tile(E, (len(t), 1)), zero, zero, theta5_abs, (l51 + l52)/2, dtheta5, ddtheta5)
cg_pos['6'], cg_vel['6'], cg_acc['6'] = point_from_base(G_arr, G_vel, G_acc, theta6_abs, r6/2, dtheta6, ddtheta6)
cg_pos['7'], cg_vel['7'], cg_acc['7'] = point_from_base(H_arr, H_vel, H_acc, theta7_abs, (l71 + l72)/2, dtheta7, ddtheta7)

body_omega['1'], body_alpha['1'] = dtheta1, ddtheta1
body_omega['2'], body_alpha['2'] = dtheta2, ddtheta2
body_omega['4'], body_alpha['4'] = dtheta4, ddtheta4
body_omega['3'], body_alpha['3'] = dtheta3, ddtheta3
body_omega['5'], body_alpha['5'] = dtheta5, ddtheta5
body_omega['6'], body_alpha['6'] = dtheta6, ddtheta6
body_omega['7'], body_alpha['7'] = dtheta7, ddtheta7

print('Dynamische parameters gedefinieerd.')
for name in ['1', '2', '4', '3', '5', '6', '7']:
    print(f"link {name}: m = {body_mass[name]:.3f} kg, I_cg = {body_inertia[name]:.5f} kg m^2")
print(f'F_contact_end = {F_contact_end} N')

In [ ]:
unknowns = [
    'F_Ax', 'F_Ay',
    'F_Bx', 'F_By',
    'F_Cx', 'F_Cy',
    'F_Dx', 'F_Dy',
    'F_Ex', 'F_Ey',
    'F_Fx', 'F_Fy',
    'F_G3x', 'F_G3y',
    'F_G6x', 'F_G6y',
    'F_Hx', 'F_Hy',
    'F_Ix', 'F_Iy',
    'M_A',
]

force_col = {
    'A': (0, 1),
    'B': (2, 3),
    'C': (4, 5),
    'D': (6, 7),
    'E': (8, 9),
    'F': (10, 11),
    'G3': (12, 13),
    'G6': (14, 15),
    'H': (16, 17),
    'I': (18, 19),
}
M_A_col = 20

solutions = np.zeros((len(t), len(unknowns)))
dyn_residual_norm = np.zeros(len(t))
cond_A_dyn = np.zeros(len(t))


def add_body_equations(A_dyn, B_dyn, row, k, body, joints, known_force=None, known_moment=0.0, input_moment=False):
    if known_force is None:
        known_force = np.zeros(2)

    m = body_mass[body]
    I_cg = body_inertia[body]
    acc = cg_acc[body][k]
    alpha = body_alpha[body][k]
    cg = cg_pos[body][k]

    # Krachtenvergelijking in x en y.
    for joint_name, point, sign in joints:
        col_x, col_y = force_col[joint_name]
        A_dyn[row, col_x] += sign
        A_dyn[row + 1, col_y] += sign

    B_dyn[row] = m*acc[0] - known_force[0]
    B_dyn[row + 1] = m*acc[1] - known_force[1]

    # Momentvergelijking rond het massacentrum.
    for joint_name, point, sign in joints:
        col_x, col_y = force_col[joint_name]
        r = point - cg
        A_dyn[row + 2, col_x] += -sign*r[1]
        A_dyn[row + 2, col_y] += sign*r[0]

    if input_moment:
        A_dyn[row + 2, M_A_col] += 1.0

    B_dyn[row + 2] = I_cg*alpha - known_moment
    return row + 3

for k in range(len(t)):
    A_dyn = np.zeros((len(unknowns), len(unknowns)))
    B_dyn = np.zeros(len(unknowns))
    row = 0

    weight = {name: np.array([0.0, -body_mass[name]*g]) for name in body_mass}

    row = add_body_equations(
        A_dyn, B_dyn, row, k, '1',
        [('A', A, +1.0), ('C', C_arr[k], +1.0)],
        known_force=weight['1'],
        input_moment=True,
    )
    row = add_body_equations(
        A_dyn, B_dyn, row, k, '2',
        [('B', B, +1.0), ('D', D_arr[k], +1.0)],
        known_force=weight['2'],
    )
    row = add_body_equations(
        A_dyn, B_dyn, row, k, '4',
        [('C', C_arr[k], -1.0), ('D', D_arr[k], -1.0), ('G3', G_arr[k], -1.0), ('G6', G_arr[k], -1.0)],
        known_force=weight['4'],
    )
    row = add_body_equations(
        A_dyn, B_dyn, row, k, '3',
        [('G3', G_arr[k], +1.0), ('F', F_arr[k], +1.0)],
        known_force=weight['3'],
    )
    row = add_body_equations(
        A_dyn, B_dyn, row, k, '5',
        [('E', E, +1.0), ('F', F_arr[k], -1.0), ('H', H_arr[k], +1.0)],
        known_force=weight['5'],
    )
    row = add_body_equations(
        A_dyn, B_dyn, row, k, '6',
        [('G6', G_arr[k], +1.0), ('I', I_arr[k], +1.0)],
        known_force=weight['6'],
    )

    known_force_7 = weight['7'] + F_contact_end
    known_moment_7 = np.cross(np.append(End_arr[k] - cg_pos['7'][k], 0.0), np.append(F_contact_end, 0.0))[2]
    row = add_body_equations(
        A_dyn, B_dyn, row, k, '7',
        [('H', H_arr[k], -1.0), ('I', I_arr[k], -1.0)],
        known_force=known_force_7,
        known_moment=known_moment_7,
    )

    if row != len(unknowns):
        raise RuntimeError(f'Verwacht 21 vergelijkingen, kreeg {row}.')

    x = np.linalg.lstsq(A_dyn, B_dyn, rcond=None)[0]
    solutions[k] = x
    dyn_residual_norm[k] = np.linalg.norm(A_dyn @ x - B_dyn)
    cond_A_dyn[k] = np.linalg.cond(A_dyn)

# Resultaten per onbekende beschikbaar maken.
result = {name: solutions[:, i] for i, name in enumerate(unknowns)}
M_A = result['M_A']
P_A = M_A*dtheta1

joint_force_mag = {}
for joint_name, (col_x, col_y) in force_col.items():
    joint_force_mag[joint_name] = np.hypot(solutions[:, col_x], solutions[:, col_y])

print('Inverse dynamica opgelost met A_dyn w = B_dyn.')
print(f'- matrixgrootte = {len(unknowns)} x {len(unknowns)}')
print(f'- max residu dynamisch stelsel = {dyn_residual_norm.max():.3e}')
print(f'- max conditiegetal A_dyn = {cond_A_dyn.max():.3e}')

## 3. Resultaten en controles

De belangrijkste resultaten zijn het aandrijfmoment `M_A`, het inputvermogen `M_A*dtheta1` en de reactiekrachten in de gewrichten.

De energiebalans controleert de matrixmethode onafhankelijk:

$$
M_A \dot\theta_1 + F_{contact}\cdot v_{End} \approx \frac{d}{dt}(T+V).
$$

Als `F_contact_end = [0, 0]`, valt de contactterm weg.

In [ ]:
# Energiecontrole.
kinetic_energy = np.zeros(len(t))
potential_energy = np.zeros(len(t))
for body in ['1', '2', '4', '3', '5', '6', '7']:
    v2 = np.sum(cg_vel[body]**2, axis=1)
    kinetic_energy += 0.5*body_mass[body]*v2 + 0.5*body_inertia[body]*body_omega[body]**2
    potential_energy += body_mass[body]*g*cg_pos[body][:, 1]

total_energy = kinetic_energy + potential_energy
energy_rate = np.gradient(total_energy, Ts, edge_order=2)
contact_power = End_vel @ F_contact_end
power_balance_lhs = P_A + contact_power

mask = np.abs(dtheta1) > 0.02
power_balance_rms = np.sqrt(np.mean((power_balance_lhs[mask] - energy_rate[mask])**2))

idx_tau_max = int(np.argmax(M_A))
idx_tau_min = int(np.argmin(M_A))
max_force_joint = max(joint_force_mag, key=lambda name: joint_force_mag[name].max())

fig, axs = plt.subplots(2, 2, figsize=(14, 9))

axs[0, 0].plot(t, M_A, lw=2)
axs[0, 0].axhline(0.0, color='black', lw=1, alpha=0.4)
axs[0, 0].set_title('Aandrijfmoment rond A')
axs[0, 0].set_xlabel('tijd [s]')
axs[0, 0].set_ylabel('M_A [N m]')
axs[0, 0].grid(True, alpha=0.3)

axs[0, 1].plot(t, P_A, label='motorvermogen M_A*dtheta1', lw=2)
axs[0, 1].plot(t, contact_power, label='contactvermogen', lw=2)
axs[0, 1].set_title('Vermogenstermen')
axs[0, 1].set_xlabel('tijd [s]')
axs[0, 1].set_ylabel('vermogen [W]')
axs[0, 1].legend()
axs[0, 1].grid(True, alpha=0.3)

for name, values in joint_force_mag.items():
    axs[1, 0].plot(t, values, label=name, lw=1.4)
axs[1, 0].set_title('Grootte van de gewrichtsreacties')
axs[1, 0].set_xlabel('tijd [s]')
axs[1, 0].set_ylabel('|F| [N]')
axs[1, 0].legend(ncol=2, fontsize=8)
axs[1, 0].grid(True, alpha=0.3)

axs[1, 1].plot(t, power_balance_lhs, label='M_A*dtheta1 + F_contact*v_End', lw=2)
axs[1, 1].plot(t, energy_rate, '--', label='d/dt(T+V)', lw=2)
axs[1, 1].set_title('Energetische controle')
axs[1, 1].set_xlabel('tijd [s]')
axs[1, 1].set_ylabel('vermogen [W]')
axs[1, 1].legend()
axs[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print('Samenvatting inverse dynamica')
print(f'- minimum M_A = {M_A[idx_tau_min]:.4f} N m op t = {t[idx_tau_min]:.2f} s')
print(f'- maximum M_A = {M_A[idx_tau_max]:.4f} N m op t = {t[idx_tau_max]:.2f} s')
print(f'- grootste gewrichtsreactie zit in {max_force_joint}: {joint_force_mag[max_force_joint].max():.2f} N')
print(f'- RMS fout energiebalans = {power_balance_rms:.3e} W')
print(f'- max dynamisch matrixresidu = {dyn_residual_norm.max():.3e}')

In [ ]:
# Compacte tabel met piekwaarden per gewricht.
print('Piekreacties per gewricht')
for name in force_col:
    idx = int(np.argmax(joint_force_mag[name]))
    print(f'{name:>2s}: max |F| = {joint_force_mag[name][idx]:8.2f} N op t = {t[idx]:5.2f} s')

## 4. Wat dit toevoegt tegenover de vereenvoudigde inverse dynamica

De hoofdnotebook bevat al een compacte energiemethode voor het benodigde inputmoment. Deze notebook volgt de Les-3-structuur explicieter:

- per bewegend gelid worden `sum Fx`, `sum Fy` en `sum M_cg` opgesteld;
- alle gewrichtsreacties zitten als onbekenden in een lineair stelsel;
- het aandrijfmoment `M_A` wordt tegelijk met de reactiekrachten opgelost;
- zwaartekracht en een optionele eindcontactkracht zitten als externe krachten in `B_dyn`.

Daarmee is dit een volledige inverse-dynamica implementatie in de stijl van de voorbeeldnotebooks.